# Predicting and Preventing Student Dropout with Apache Spark

**Final Project: Business Data Analytics with Apache Spark**

## Business problem

> **With a limited support budget, how can a university identify the students most at risk of dropping out as early as the first semester, and which levers (scholarships, student work) should it act on first?**

- **Decision-maker:** the academic affairs office and the student success department.
- **Decision:** which students to support first (tutoring, follow-up), and where to invest (scholarships, on-campus jobs).
- **Constraint:** the budget does not allow supporting every student, for example only 20% of them.
- **Success metric:** the share of future dropouts reached when targeting that 20%.

## Notebook outline

1. Spark setup and data loading
2. Data quality
3. Exploration with Spark SQL: who drops out?
4. Prediction model with Spark MLlib (risk score)
5. Targeting with a limited budget
6. Action levers: scholarships and student work
7. Conclusions and recommendations

## Dataset

`student_dropout_risk_2026.csv`: 100,000 students, 34 columns. The target `dropout` equals 1 if the student drops out.

## 1. Spark setup and data loading

The notebook runs **identically on Databricks (Free Edition, serverless compute) and locally**:

- on Databricks, `spark` already exists and `getOrCreate()` retrieves it;
- locally, `getOrCreate()` creates a new Spark session.

The code only uses the DataFrame API, Spark SQL and `pyspark.ml`. It uses neither RDDs nor `cache()`, which are not available on serverless compute.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("student-dropout").getOrCreate()
print("Spark version:", spark.version)

# Fixed random seed: the random train/test split and the Random Forest
# give the same results at every run (reproducibility)
SEED = 42

In [ ]:
# --- Path to the data file ---------------------------------------------------
# Locally: the CSV sits in the same folder as the notebook.
# On Databricks: the CSV is uploaded to a Unity Catalog volume
#   (Catalog > workspace > default > Create > Volume "projet_spark" > Upload).
#   Change DATABRICKS_PATH if the volume has another name.
LOCAL_PATH = "student_dropout_risk_2026.csv"
DATABRICKS_PATH = "/Volumes/workspace/default/projet_spark/student_dropout_risk_2026.csv"

DATA_PATH = os.path.abspath(LOCAL_PATH) if os.path.exists(LOCAL_PATH) else DATABRICKS_PATH
print("Reading:", DATA_PATH)

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)   # let Spark detect column types
      .csv(DATA_PATH))

df.printSchema()

## 2. Data quality

Before any analysis, we check the number of rows, missing values, duplicates and the distribution of the target.

In [ ]:
n_rows, n_cols = df.count(), len(df.columns)
print(f"{n_rows:,} rows, {n_cols} columns")

# Missing values per column (only columns with at least one missing value are counted)
nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T
nulls.columns = ["missing_values"]
print("Columns with missing values:", int((nulls["missing_values"] > 0).sum()))

# Duplicates (on all columns except the ID)
n_dup = n_rows - df.drop("person_id").dropDuplicates().count()
print("Duplicates:", n_dup)

In [ ]:
# Target distribution
df.groupBy("dropout").count().withColumn("share_%", F.round(100 * F.col("count") / n_rows, 1)) \
  .orderBy("dropout").show()

GLOBAL_RATE = 100 * df.agg(F.avg("dropout")).first()[0]
print(f"Overall dropout rate: {GLOBAL_RATE:.1f}%")

In [ ]:
# Descriptive statistics for a few key variables
df.select("age", "gpa_semester_1", "gpa_semester_2", "attendance_rate",
          "weekly_work_hours", "stress_level", "units_failed").describe().show()

**Finding:** the dataset is complete. There are no missing values, no duplicates, and the target is fairly balanced (about 32% dropouts). No heavy cleaning is needed.

*Limitation:* data this clean, dated 2026, is very likely **synthetic**. The relationships observed should therefore be confirmed on real university data.

## 3. Exploration with Spark SQL: who drops out?

We register the DataFrame as a temporary SQL table, then compute dropout rates by group.

In [ ]:
df.createOrReplaceTempView("students")

def rate_by(expr, alias, order=None):
    # Dropout rate (%) and headcount for each value of a variable, computed with Spark SQL.
    order = order or alias
    q = f'''
        SELECT {expr} AS {alias},
               COUNT(*) AS n_students,
               ROUND(100 * AVG(dropout), 1) AS dropout_rate
        FROM students
        GROUP BY 1
        ORDER BY {order}
    '''
    return spark.sql(q)

### 3.1 Academic results: first-semester GPA

In [ ]:
gpa = rate_by('''CASE WHEN gpa_semester_1 < 2.0 THEN '1. < 2.0'
                      WHEN gpa_semester_1 < 2.5 THEN '2. 2.0-2.5'
                      WHEN gpa_semester_1 < 3.0 THEN '3. 2.5-3.0'
                      WHEN gpa_semester_1 < 3.5 THEN '4. 3.0-3.5'
                      ELSE '5. >= 3.5' END''', "gpa_s1_band")
gpa.show()

gpa_pd = gpa.toPandas()
plt.bar(gpa_pd["gpa_s1_band"], gpa_pd["dropout_rate"])
plt.title("Dropout rate by first-semester GPA")
plt.xlabel("First-semester GPA")
plt.ylabel("Dropout rate (%)")
plt.show()

### 3.2 Engagement: class attendance

In [ ]:
att = rate_by('''CASE WHEN attendance_rate < 60 THEN '1. < 60%'
                      WHEN attendance_rate < 70 THEN '2. 60-70%'
                      WHEN attendance_rate < 80 THEN '3. 70-80%'
                      WHEN attendance_rate < 90 THEN '4. 80-90%'
                      ELSE '5. >= 90%' END''', "attendance_band")
att.show()

### 3.3 Financial and social situation

In [ ]:
for col in ["family_income_level", "scholarship_holder", "tuition_up_to_date",
            "first_generation", "part_time_job"]:
    rate_by(col, col).show()

### 3.4 Well-being: stress

In [ ]:
rate_by("stress_level", "stress").show()

### 3.5 What barely matters

In [ ]:
for col in ["course_type", "gender", "study_mode", "international_student"]:
    rate_by(col, col).show()

### 3.6 An interpretation trap: mental health support

In [ ]:
rate_by("mental_health_support", "mental_health_support").show()

**Exploration summary**

- **Academic results** (first-semester GPA, failed units) are the strongest signals.
- **Engagement** (attendance, assignments submitted, missed deadlines) also matters a lot.
- **Financial situation** weighs in: low income, unpaid tuition, a job alongside studies.
- **Well-being** follows a steady slope: the higher the stress, the more frequent the dropout.
- **Field of study, gender, study mode and international status make almost no difference** (about 32% everywhere).
- **Correlation is not causation:** students receiving mental health support drop out *more*. Support does not cause dropout: students who are struggling are the ones who seek it.

## 4. Prediction model with Spark MLlib

**Goal:** give each student a **probability of dropping out** (a risk score from 0 to 100%).

**Design choices**

- **Early warning.** To act *before* dropout happens, the main model does not use the second-semester GPA. We then compare it with the full model to measure what we lose by acting early.
- **Ethics.** Gender and nationality are excluded from the model: support should not depend on these criteria. We check below that this costs nothing in accuracy.
- **Two algorithms:** logistic regression (simple and interpretable) and Random Forest (captures non-linear effects).
- **Evaluation:** 70% training / 30% test split. Main metric: **AUC** (0.5 = random guessing, 1 = perfect).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

CAT_COLS = ["family_income_level", "course_type", "study_mode"]   # text (categorical) variables
SENSITIVE = ["gender", "nationality"]                               # excluded (ethics)
EXCLUDED = ["person_id", "dropout"] + SENSITIVE + CAT_COLS
NUM_ALL = [c for c in df.columns if c not in EXCLUDED]               # numeric variables
NUM_EARLY = [c for c in NUM_ALL if c != "gpa_semester_2"]            # without semester 2

print(len(NUM_EARLY), "numeric variables (early warning) +", len(CAT_COLS), "categorical variables")

# Random 70/30 split, made reproducible by the seed
train, test = df.randomSplit([0.7, 0.3], seed=SEED)
print("Training:", train.count(), "| Test:", test.count())

In [ ]:
def build_pipeline(num_cols, cat_cols, model):
    # Feature preparation + model, chained in a single Spark ML Pipeline.
    # 1) StringIndexer: text categories -> numeric indexes
    idx = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
    # 2) OneHotEncoder: indexes -> 0/1 indicator columns
    ohe = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols],
                        outputCols=[c + "_ohe" for c in cat_cols])
    # 3) VectorAssembler: all variables -> one feature vector per student
    assembler = VectorAssembler(inputCols=num_cols + [c + "_ohe" for c in cat_cols],
                                outputCol="features_raw")
    # 4) StandardScaler: put all variables on the same scale
    scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                            withMean=True, withStd=True)
    return Pipeline(stages=idx + [ohe, assembler, scaler, model])

evaluator = BinaryClassificationEvaluator(labelCol="dropout", rawPredictionCol="rawPrediction",
                                          metricName="areaUnderROC")

def evaluate(model, data, name):
    # AUC + accuracy, precision and recall at a 0.5 threshold on the test set.
    pred = model.transform(data)
    auc = evaluator.evaluate(pred)
    m = pred.agg(
        F.sum(((F.col("prediction") == 1) & (F.col("dropout") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("prediction") == 1) & (F.col("dropout") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("prediction") == 0) & (F.col("dropout") == 1)).cast("int")).alias("fn"),
        F.count("*").alias("n"),
        F.sum((F.col("prediction") == F.col("dropout")).cast("int")).alias("ok"),
    ).first()
    return {"model": name, "AUC": round(auc, 3),
            "accuracy": round(m.ok / m.n, 3),
            "precision": round(m.tp / (m.tp + m.fp), 3),
            "recall": round(m.tp / (m.tp + m.fn), 3)}

lr = LogisticRegression(labelCol="dropout", featuresCol="features", maxIter=100)
rf = RandomForestClassifier(labelCol="dropout", featuresCol="features",
                            numTrees=60, maxDepth=8, seed=SEED)   # limited size (< 100 MB on serverless)

In [ ]:
# Each model is evaluated right after training. Models that are not reused later
# are deleted to free memory (the ML model cache is limited on serverless compute).
results = []

# 1) Logistic regression - early warning (kept: used in sections 4.1 and 5)
lr_early = build_pipeline(NUM_EARLY, CAT_COLS, lr).fit(train)
results.append(evaluate(lr_early, test, "Logistic regression - early warning"))

# 2) Random Forest - early warning
model = build_pipeline(NUM_EARLY, CAT_COLS, rf).fit(train)
results.append(evaluate(model, test, "Random Forest - early warning"))
del model

# 3) Logistic regression - full model (with second-semester GPA)
model = build_pipeline(NUM_ALL, CAT_COLS, lr).fit(train)
results.append(evaluate(model, test, "Logistic regression - full (with semester 2)"))
del model

# 4) Ethics check: early warning WITH gender and nationality
model = build_pipeline(NUM_EARLY, CAT_COLS + SENSITIVE, lr).fit(train)
results.append(evaluate(model, test, "Check: early warning + gender and nationality"))
del model

results = pd.DataFrame(results)
results

**Reading the results**

- Both early-warning models separate profiles very well. **An AUC of about 0.90** means that if we pick a dropout and a non-dropout at random, the model gives the dropout a higher score in about 90% of cases.
- **Adding the second-semester GPA brings almost nothing.** The university can therefore act as early as the first semester without losing prediction quality.
- **Adding gender and nationality does not improve the model.** Excluding them is an ethical choice that costs nothing.
- Logistic regression even does slightly better than Random Forest (AUC 0.90 vs 0.88), while being easier to explain: **it is the model we keep**.

### 4.1 Which variables matter most?

Variables were standardized, so the logistic regression coefficients are comparable: a positive coefficient increases the risk, a negative one reduces it.

In [ ]:
lr_model = lr_early.stages[-1]
assembler_inputs = lr_early.stages[-3].getInputCols()

# Variable names after encoding (categorical variables are split into indicators)
names = []
idx_models = [s for s in lr_early.stages if hasattr(s, "labels")]
for c in assembler_inputs:
    if c.endswith("_ohe"):
        base = c[:-4]
        labels = next(m.labels for m in idx_models if m.getOutputCol() == base + "_idx")
        names += [f"{base}={l}" for l in labels]          # dropLast: the last category is the reference
    else:
        names.append(c)

coefs = lr_model.coefficients.toArray()
coef_pdf = (pd.DataFrame({"variable": names[:len(coefs)], "coefficient": coefs})
            .assign(abs=lambda d: d.coefficient.abs())
            .sort_values("abs", ascending=False).drop(columns="abs").head(12))
coef_pdf.round(3)

## 5. Targeting with a limited budget

**Business question:** if the university can only support part of its students, how many future dropouts does it reach by targeting those the model considers most at risk?

We rank test-set students from highest to lowest risk, then split them into 20 bands of 5%.

In [ ]:
scored = (lr_early.transform(test)
          .withColumn("p_dropout", vector_to_array("probability")[1])   # probability of class 1
          .select("person_id", "dropout", "p_dropout", "family_income_level", "scholarship_holder",
                  "part_time_job", "weekly_work_hours", "gpa_semester_1", "attendance_rate",
                  "stress_level", "tuition_up_to_date", "units_failed"))

# 5% band (1 = the 5% most at risk)
# Note: a window without partition moves the data to a single partition.
# This is harmless here (30,000 rows) but should be avoided on much larger volumes.
w = Window.orderBy(F.desc("p_dropout"))
gain = (scored.withColumn("band", F.ntile(20).over(w))
        .groupBy("band")
        .agg(F.count("*").alias("n"), F.sum("dropout").alias("dropouts"))
        .orderBy("band").toPandas())

total_drop = gain["dropouts"].sum()
gain["pct_targeted"] = 5 * gain["band"]
gain["pct_dropouts_reached"] = 100 * gain["dropouts"].cumsum() / total_drop
gain["targeting_precision"] = 100 * gain["dropouts"].cumsum() / gain["n"].cumsum()

budget = gain[gain["pct_targeted"].isin([10, 20, 30, 50])][
    ["pct_targeted", "pct_dropouts_reached", "targeting_precision"]].round(1)
budget.columns = ["% of students supported", "% of dropouts reached", "% true dropouts among targeted"]
budget

In [ ]:
plt.plot(gain["pct_targeted"], gain["pct_dropouts_reached"], label="Model")
plt.plot([0, 100], [0, 100], "--", label="Random targeting")
plt.title("Gain curve")
plt.xlabel("% of students supported (highest risk first)")
plt.ylabel("% of dropouts reached")
plt.legend()
plt.show()

### 5.1 Three risk levels, three responses

We turn the probability into three simple levels that staff can use: low risk (< 30%), medium (30 to 60%) and high (> 60%).

In [ ]:
tiers = scored.withColumn("risk_level",
    F.when(F.col("p_dropout") >= 0.6, "3. high")
     .when(F.col("p_dropout") >= 0.3, "2. medium")
     .otherwise("1. low"))

profile = (tiers.groupBy("risk_level").agg(
        F.count("*").alias("n_students"),
        F.round(100 * F.count("*") / F.lit(scored.count()), 1).alias("share_%"),
        F.round(100 * F.avg("dropout"), 1).alias("actual_dropout_%"),
        F.round(F.avg("gpa_semester_1"), 2).alias("avg_gpa_s1"),
        F.round(F.avg("attendance_rate"), 1).alias("avg_attendance_%"),
        F.round(100 * F.avg((F.col("family_income_level") == "Low").cast("int")), 1).alias("low_income_%"),
        F.round(100 * F.avg("part_time_job"), 1).alias("with_job_%"),
        F.round(100 * F.avg(1 - F.col("tuition_up_to_date")), 1).alias("unpaid_tuition_%"),
        F.round(F.avg("stress_level"), 1).alias("avg_stress"))
    .orderBy("risk_level"))
profile.toPandas()

The high-risk group looks very different from the rest: lower grades, lower attendance, and more students with low income, a job and unpaid tuition. This points directly to the action levers.

## 6. Action levers

The score tells us **who** to support. We still need to know **how**. We study two levers funded by the university.

### 6.1 Scholarships: who should get them first?

We compare dropout among scholarship holders and non-holders **at the same income level**.

In [ ]:
schol = spark.sql('''
    SELECT family_income_level AS income,
           ROUND(100 * AVG(CASE WHEN scholarship_holder = 0 THEN dropout END), 1) AS dropout_without_scholarship,
           ROUND(100 * AVG(CASE WHEN scholarship_holder = 1 THEN dropout END), 1) AS dropout_with_scholarship,
           SUM(CASE WHEN scholarship_holder = 0 THEN 1 ELSE 0 END)              AS n_without_scholarship
    FROM students
    GROUP BY family_income_level
    ORDER BY CASE family_income_level WHEN 'Low' THEN 1 WHEN 'Middle' THEN 2 ELSE 3 END
''').toPandas()
schol["gap_points"] = (schol["dropout_without_scholarship"] - schol["dropout_with_scholarship"]).round(1)
schol

In [ ]:
schol.plot(x="income", y=["dropout_without_scholarship", "dropout_with_scholarship"], kind="bar")
plt.title("Dropout rate with and without scholarship, by income level")
plt.ylabel("Dropout rate (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Order of magnitude: dropouts avoided if N low-income students without a scholarship receive one.
# Simple assumption: the observed gap applies as is (see the limitation below).
low = schol[schol["income"] == "Low"].iloc[0]
for n in [500, 1000, 2000]:
    print(f"{n:>5} targeted scholarships -> about {n * low['gap_points'] / 100:.0f} dropouts avoided")
print(f"\nLow-income students without a scholarship: {int(low['n_without_scholarship']):,}")

**Reading:** scholarships are associated with lower dropout in every group, but **the effect is about twice as strong among low-income students**. With the same budget, it is therefore more efficient to target scholarships at these students.

*Limitation:* this is an observed comparison, not an experiment. Scholarship holders may differ from non-holders in other ways. The estimate of dropouts avoided is an order of magnitude, not a forecast.

### 6.2 Student work: when does it become a risk?

We look at dropout by band of weekly working hours, **separately for each income level**. This checks that the effect of work is not just a hidden effect of income.

In [ ]:
hours_expr = '''CASE WHEN weekly_work_hours = 0 THEN '0 h'
                     WHEN weekly_work_hours <= 10 THEN '1-10 h'
                     WHEN weekly_work_hours <= 20 THEN '11-20 h'
                     WHEN weekly_work_hours <= 30 THEN '21-30 h'
                     ELSE '31-40 h' END'''

job = rate_by(hours_expr, "work_hours")
job.show()

job_pd = job.toPandas()
plt.bar(job_pd["work_hours"], job_pd["dropout_rate"])
plt.title("Dropout rate by weekly working hours")
plt.xlabel("Working hours per week")
plt.ylabel("Dropout rate (%)")
plt.show()

In [ ]:
# Same analysis split by income level (pivot table)
job_income = (spark.sql(f'''
    SELECT {hours_expr} AS work_hours, family_income_level AS income, dropout
    FROM students''')
    .groupBy("work_hours").pivot("income", ["Low", "Middle", "High"])
    .agg(F.round(100 * F.avg("dropout"), 1))
    .orderBy("work_hours"))
job_income.show()

**Reading:** dropout rises **gradually** with working hours: there is no sharp threshold. The increase appears **at every income level**, so work itself weighs on student success, beyond the financial situation. Beyond 20 hours per week, the risk becomes particularly high.

## 7. Conclusions and recommendations

### Answer to the business problem

*The figures below come from our run. They may vary very slightly from one environment to another (random data split).*

1. **At-risk students can be identified early.** A logistic regression trained with Spark MLlib, using only data available in the first semester, reaches an **AUC of 0.90**. Adding second-semester grades barely improves it (0.904).
2. **A limited budget is enough to address a large part of the problem.** By supporting the **20%** of students most at risk, the university reaches **about 54% of future dropouts**, and 85% of targeted students are true dropouts. Random targeting would reach only 20%.
3. **The levers are identified:**
   - **scholarships** are associated with 9 points less dropout among low-income students (49% vs 40%), versus 4 points among high-income students: they should be concentrated there. 10,393 low-income students do not have one;
   - dropout rises from **25% without a job to 45% at 21-30 hours** of work per week, and the increase holds at every income level: the university should offer limited-hours on-campus jobs and flexible schedules;
   - **low grades and low attendance** call for tutoring and attendance monitoring.

### Recommendations for the university

| Risk level | Proposed action |
|---|---|
| High | One-to-one meeting, tutoring, priority scholarship review, schedule adjustment if heavy job |
| Medium | Attendance monitoring, automatic alert if attendance drops |
| Low | No specific action |

### What Spark brings

- Processes the 100,000 students in seconds, with code that would scale to millions of rows (several universities, several years).
- The same processing chain (Spark SQL, then an MLlib Pipeline) can be reused every semester to recompute the scores.

### Limitations

- The data is very likely synthetic: results need to be validated on real data.
- The observed effects (scholarships, work) are **correlations**, not proven causes. An experiment (for example, granting scholarships to a test group) would measure the real effect.
- The score must be used to **help** students, never to exclude them or filter admissions.